# Financial Behavior Segmentation - Sri Lanka

This notebook explores the Global Findex data to identify distinct financial behavior segments specifically for **Sri Lanka**.

## Project Goals
1. **Segmentation**: Identify distinct segments based on financial behavior using Clustering.
2. **Dimensionality Reduction**: Use PCA and Factor Analysis to reduce the large number of survey variables into interpretable financial behaviors.
3. **Profiling**: Characterize each segment.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings

warnings.filterwarnings('ignore')

# Try importing FactorAnalysis related libraries
try:
    from factor_analyzer import FactorAnalyzer
    from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
except ImportError:
    print("factor_analyzer not installed. Please install specific library if needed: pip install factor-analyzer")

# Configuration
sns.set(style="whitegrid")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Data Loading and Cleaning

In [ ]:
# Load the PCA-ready dataset (imputed)
file_path = 'findex_microdata_2025_pca.csv'
df = pd.read_csv(file_path)

# Filter for Sri Lanka if not already done, though file name suggests it might be processed
if 'economy' in df.columns:
    df_sl = df[df['economy'] == 'Sri Lanka'].copy()
else:
    df_sl = df.copy()

print(f"Data Shape: {df_sl.shape}")
df_sl.head()

## 2. Dimensionality Reduction (PCA & Factor Analysis)

We have many variables related to financial usage (`fin*`) and confidence/capability (`con*`). 
We want to reduce these to a smaller set of latent factors that explain financial behavior.

In [ ]:
# 2.1 Feature Selection
# Exclude metadata and demographics for the *reduction* step to derive purely behavioral factors.
excluded_cols = ['year', 'economy', 'economycode', 'regionwb', 'pop_adult', 'wpid_random', 'wgt', 
                 'female', 'age', 'educ', 'inc_q', 'emp_in', 'urbanicity']

# Select only numeric columns roughly corresponding to survey responses
features_for_pca = [c for c in df_sl.columns if c not in excluded_cols and df_sl[c].dtype in ['int64', 'float64']]

print(f"Number of features selected for reduction: {len(features_for_pca)}")

X = df_sl[features_for_pca]

# 2.2 Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for zero variance features (constants)
selector = VarianceThreshold(threshold=0)
selector.fit(X_scaled)
X_scaled_clean = X_scaled[:, selector.get_support()]
selected_features_names = np.array(features_for_pca)[selector.get_support()]

print(f"Features remaining after removing constants: {X_scaled_clean.shape[1]}")

### 2.3 Principal Component Analysis (PCA)

In [ ]:
pca = PCA()
pca.fit(X_scaled_clean)

# Calculate cumulative variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Scree Plot
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.axhline(y=0.8, color='r', linestyle='-', label='80% Explained Variance')
plt.title('PCA Scree Plot: Cumulative Variance Explained')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.show()

### 2.4 Factor Analysis (Final Model)
Based on initial analysis, ~21 factors provided a good balance.

In [ ]:
# Define number of factors (you can adjust this based on the Scree plot findings)
N_FACTORS = 21 

try:
    fa = FactorAnalyzer(n_factors=N_FACTORS, rotation='varimax')
    fa.fit(X_scaled_clean)

    # Get loadings
    fa_loadings = pd.DataFrame(fa.loadings_, 
                               index=selected_features_names, 
                               columns=[f'Factor{i+1}' for i in range(N_FACTORS)])
    
    # Visualize Loadings - OPTIONAL: See top contributing features for each factor
    print("Top 3 Loading Features for each Factor:")
    for col in fa_loadings.columns:
        top_features = fa_loadings[col].abs().sort_values(ascending=False).head(3)
        print(f"{col}: {', '.join([f'{idx} ({val:.2f})' for idx, val in zip(top_features.index, top_features.values)])}")

    # 3. Extract Factor Scores for Clustering
    X_factors = fa.transform(X_scaled_clean)
    print(f"Factor Scores Shape: {X_factors.shape}")

except NameError:
    print("FactorAnalyzer not installed. Using PCA components instead for demonstration.")
    pca_final = PCA(n_components=N_FACTORS)
    X_factors = pca_final.fit_transform(X_scaled_clean)
    print(f"PCA Scores Shape: {X_factors.shape}")

## 3. Clustering Segments
We will specificy 'k' clusters using K-Means on the Factor Scores.

In [ ]:
# 3.1 Determine Optimal k (Elbow Method & Silhouette)
wcss = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_factors)
    wcss.append(kmeans.inertia_)
    score = silhouette_score(X_factors, kmeans.labels_)
    silhouette_scores.append(score)

# Plotting
fig, ax1 = plt.subplots(figsize=(12, 6))

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('WCSS (Inertia)', color=color)
ax1.plot(K_range, wcss, color=color, marker='o', label='WCSS')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Silhouette Score', color=color)
ax2.plot(K_range, silhouette_scores, color=color, marker='s', linestyle='--', label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Elbow Method & Silhouette Score for Optimal k')
plt.show()

### 3.2 Fit Final Cluster Model
Based on analysis (Silhouette Score peak), we select **k=7**.

In [ ]:
# Set chosen number of clusters here
final_k = 7 

kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
df_sl['Cluster'] = kmeans_final.fit_predict(X_factors)

print("Cluster Counts:")
print(df_sl['Cluster'].value_counts().sort_index())

## 4. Cluster Profiling
Understanding who is in each cluster.

In [ ]:
# 4.1 Profiling with Demographics
# Create a summary table grouped by Cluster
demographic_cols = ['age', 'educ', 'inc_q', 'female', 'emp_in', 'urbanicity']
profile_summary = df_sl.groupby('Cluster')[demographic_cols].mean()

print("Demographic Profile (Mean values):")
display(profile_summary)

# 4.2 Visualizing Financial Behaviors by Cluster (Heatmap)
# We can look at the average factor scores for each cluster to see what drives them
cluster_factor_means = pd.DataFrame(X_factors, columns=[f'Factor{i+1}' for i in range(X_factors.shape[1])])
cluster_factor_means['Cluster'] = df_sl['Cluster'].values
heatmap_data = cluster_factor_means.groupby('Cluster').mean()

plt.figure(figsize=(15, 8))
sns.heatmap(heatmap_data, cmap='RdBu', center=0, annot=True, fmt='.2f')
plt.title('Average Factor Scores by Cluster (Z-scores)')
plt.xlabel('Financial Factors')
plt.ylabel('Cluster')
plt.show()

## 5. Export and Finalization
Saving the labelled dataset with targeted segment names.

In [ ]:
# Manual Naming of Clusters based on Factor Loading Analysis
# Derived from inspecting heatmap/demographics:
# Cluster 0: Low Factor 6 (Transfers) & Factor 5 (Accounts) -> 'Non-Recipients/Vulnerable'
# Cluster 1: Low Factor 5 (Accounts) -> 'The Unbanked'
# Cluster 2: Average -> 'Mainstream Users'
# Cluster 3: Low Factor 2 (Merchant Pay) but High Income -> 'Wealthy Traditionalists'
# Cluster 4: High Factor 2 -> 'Digital Elites'
# Cluster 5: Low Factor 10 (Ag) -> 'Non-Ag Workers'
# Cluster 6: Age 68, Low Factor 8 (Pensions) -> 'Senior Citizens'

cluster_names = {
    0: 'Non-Recipients (Low Transfers)',
    1: 'The Unbanked',
    2: 'Mainstream Users',
    3: 'Wealthy Traditionalists',
    4: 'Digital Elites',
    5: 'Non-Agricultural Workers',
    6: 'Senior Citizens'
}

print("Cluster Names:")
for k, v in cluster_names.items():
    print(f"Cluster {k}: {v}")

# Assign to DataFrame
df_sl['Cluster_Name'] = df_sl['Cluster'].map(cluster_names)

# Save the clustered data
output_file = 'findex_sl_clusters.csv'
df_sl.to_csv(output_file, index=False)
print(f"\nSuccessfully saved labelled data to {output_file}")

# Save Profile Summary
profile_summary.to_csv('cluster_demographics.csv')
print("Saved 'cluster_demographics.csv'.")